In [58]:
import xarray as xr
from scipy.interpolate import RegularGridInterpolator
import pandas as pd

from scipy.spatial import cKDTree
import numpy as np

crust age

In [59]:
age = pd.read_csv('..\\data\\age_hf\\seg5_lat_long_age.txt', sep='\t', header=None, names=['lat', 'lon', 'age'])
print(age)

      lat    lon        age
0    43.0 -128.0   2.448250
1    43.0 -127.9   2.179997
2    43.0 -127.8   2.002510
3    43.0 -127.7   1.876630
4    43.0 -127.6   1.641777
..    ...    ...        ...
989  40.0 -124.9  15.446379
990  40.0 -124.8  14.237822
991  40.0 -124.7  15.990977
992  40.0 -124.6  14.295403
993  40.0 -124.5  15.872827

[994 rows x 3 columns]


heat flow

In [60]:
hf = pd.read_csv("..\\data\\age_hf\\seg5_lat_long_q.txt", sep='\t', header=None, names=['lat', 'lon', 'q'])
print(hf)

        lat       lon       q
0   40.7490 -127.4000   225.0
1   40.7440 -127.5400   299.0
2   40.7370 -127.5500  1656.0
3   40.7370 -127.5700   576.0
4   40.7500 -127.4000   240.0
..      ...       ...     ...
89  40.9967 -127.4910  2726.0
90  41.5000 -126.5333   242.0
91  40.6000 -127.4167   233.0
92  40.5167 -126.5167   163.0
93  40.9333 -126.5167   193.0

[94 rows x 3 columns]


interpolate age at each heat flow point (bc we have more age than heat flow)

In [61]:
age_ds = age.set_index(['lat', 'lon'])['age'].to_xarray()

interp = age_ds.interp(lon=xr.DataArray(hf['lon'].values), lat=xr.DataArray(hf['lat'].values), method='nearest')

hf['age'] = interp.values

print(hf)

        lat       lon       q           age
0   40.7490 -127.4000   225.0  1.465970e-01
1   40.7440 -127.5400   299.0  6.296330e-07
2   40.7370 -127.5500  1656.0  2.272070e-01
3   40.7370 -127.5700   576.0  2.272070e-01
4   40.7500 -127.4000   240.0  1.465970e-01
..      ...       ...     ...           ...
89  40.9967 -127.4910  2726.0  2.139138e-02
90  41.5000 -126.5333   242.0  2.993756e+00
91  40.6000 -127.4167   233.0  1.742110e-01
92  40.5167 -126.5167   163.0  3.889813e+00
93  40.9333 -126.5167   193.0  3.269613e+00

[94 rows x 4 columns]


but interp is not good because fills with NaN...

In [62]:
tree = cKDTree(age[['lat', 'lon']].values)
dist, idx = tree.query(hf[['lat', 'lon']].values, k=1)
hf['age'] = age['age'].values[idx]

print(hf[130:140])

Empty DataFrame
Columns: [lat, lon, q, age]
Index: []


In [ ]:
hf.to_csv('..\\created_data\\segments\\kd_sg5_q_age.csv', index=False)